# Boston Tech Week 2026: LLM Quantization Workshop

**Duration:** 60 minutes  
**Objective:** Learn about LLM quantization and measure performance improvements

In this notebook, you'll:
1. Understand what quantization is and why it matters
2. Benchmark an original FP16 model
3. Benchmark a quantized INT4 model
4. Compare the results and analyze the trade-offs

## Part 1: What is Quantization?

Model quantization reduces the precision of model weights to make inference faster and more memory-efficient:

- **FP16/BF16** (16-bit floating point) - Standard precision
- **FP8** (8-bit floating point) - 2x memory reduction
- **NVFP4** (4-bit NVIDIA floating point) - 4x memory reduction, much faster
- **INT8** (8-bit integer) - 2x memory reduction, faster compute
- **INT4** (4-bit integer) - 4x memory reduction

### Why Quantize?

1. **Faster inference** - 1.5-2.5x throughput improvement
2. **Lower memory** - 2-4x VRAM reduction (fit bigger models)
3. **Lower latency** - 30-50% faster response times
4. **Minimal quality loss** - Typically <2% degradation

### Today's Models

| Model | Precision | Size |
|-------|-----------|------|
| Qwen/Qwen3.6-35B-A3B | FP16 | ~70GB |
| RedHatAI/Qwen3.6-35B-A3B-NVFP4 | NVFP4 | ~20GB |

Both are served via vLLM with speculative decoding on the quantized model.

## Part 2: Setup

Install guidellm for benchmarking.

In [ ]:
# Install dependencies (this may take 1-2 minutes)
print("Installing guidellm, matplotlib, and pandas...")
print("Please wait...\n")
!pip install -q guidellm matplotlib pandas

print("Installation complete!")

In [ ]:
# Import libraries
import subprocess
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import os
import sys

# Add user bin to PATH (where pip installs executables)
user_bin = os.path.expanduser("~/.local/bin")
if user_bin not in os.environ["PATH"]:
    os.environ["PATH"] = f"{user_bin}:{os.environ['PATH']}"

# Model endpoints
ORIGINAL_API = "http://95.133.252.99:8000/v1"
QUANTIZED_API = "http://95.133.252.99:8001/v1"

print("Setup complete!")
print(f"Original API: {ORIGINAL_API}")
print(f"Quantized API: {QUANTIZED_API}")

# Verify guidellm is accessible
result = subprocess.run(["which", "guidellm"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"guidellm found at: {result.stdout.strip()}")
else:
    print("WARNING: guidellm not found in PATH. Installation may have failed.")
    print("Please re-run the installation cell above.")

## Part 3: Benchmark Original Model (FP16)

We'll benchmark the original Qwen/Qwen3.6-35B-A3B model running at FP16 precision.

In [ ]:
# Create output directory
!mkdir -p original

# Run benchmark
print("Benchmarking original model (FP16)...")
print("Please wait...\n")
!cd original && guidellm benchmark \
    --target "{ORIGINAL_API}" \
    --profile sweep \
    --data "prompt_tokens=100,output_tokens=100" \
    --max-requests 5 \
    --outputs json \
    >/dev/null 2>&1

print("Original model benchmark complete!")
print("Results saved to: original/benchmarks.json")

## Part 4: Benchmark Quantized Model (NVFP4)

Now let's benchmark the quantized Qwen3.6-35B-A3B model running at NVFP4 precision with speculative decoding.

In [ ]:
# Create output directory
!mkdir -p quantized

# Run benchmark
print("Benchmarking quantized model (INT4)...")
print("Please wait...\n")
!cd quantized && guidellm benchmark \
    --target "{QUANTIZED_API}" \
    --profile sweep \
    --data "prompt_tokens=100,output_tokens=100" \
    --max-requests 5 \
    --outputs json \
    >/dev/null 2>&1

print("Quantized model benchmark complete!")
print("Results saved to: quantized/benchmarks.json")

## Part 5: Compare Results

Let's load and visualize the benchmark results.

In [ ]:
# Load JSON results
try:
    with open('original/benchmarks.json', 'r') as f:
        original_data = json.load(f)
    with open('quantized/benchmarks.json', 'r') as f:
        quantized_data = json.load(f)
    
    # Get the latest benchmark run (last item in benchmarks list)
    orig = original_data['benchmarks'][-1]
    quant = quantized_data['benchmarks'][-1]
    
    # Extract metrics from the benchmark data
    orig_metrics = {
        'throughput': orig['metrics']['requests_per_second']['successful']['mean'],
        'mean_latency': orig['metrics']['request_latency']['successful']['mean'],
        'p50_latency': orig['metrics']['request_latency']['successful']['percentiles']['p50'],
        'p95_latency': orig['metrics']['request_latency']['successful']['percentiles']['p95'],
        'p99_latency': orig['metrics']['request_latency']['successful']['percentiles']['p99'],
    }
    
    quant_metrics = {
        'throughput': quant['metrics']['requests_per_second']['successful']['mean'],
        'mean_latency': quant['metrics']['request_latency']['successful']['mean'],
        'p50_latency': quant['metrics']['request_latency']['successful']['percentiles']['p50'],
        'p95_latency': quant['metrics']['request_latency']['successful']['percentiles']['p95'],
        'p99_latency': quant['metrics']['request_latency']['successful']['percentiles']['p99'],
    }
    
    print("BENCHMARK COMPARISON")
    print("=" * 80)
    print(f"\n{'Metric':<30} {'Original (FP16)':<20} {'Quantized (INT4)':<20} {'Speedup':<10}")
    print("-" * 80)
    
    metrics = ['throughput', 'mean_latency', 'p50_latency', 'p95_latency', 'p99_latency']
    
    for metric in metrics:
        orig_val = orig_metrics[metric]
        quant_val = quant_metrics[metric]
        
        # For throughput, higher is better
        if 'throughput' in metric:
            speedup = quant_val / orig_val
            print(f"{metric:<30} {orig_val:<20.2f} {quant_val:<20.2f} {speedup:.2f}x")
        # For latency, lower is better
        else:
            speedup = orig_val / quant_val
            print(f"{metric:<30} {orig_val:<20.2f} {quant_val:<20.2f} {speedup:.2f}x")
    
    print("\n" + "=" * 80)
    print("\nKey Takeaways:")
    print("  - Quantized model should be 1.5-2x faster")
    print("  - Latency should be 30-40% lower")
    print("  - Memory usage is 50-75% less (not shown in metrics)")
    
    # Store for visualization
    globals()['orig_metrics'] = orig_metrics
    globals()['quant_metrics'] = quant_metrics
    
except FileNotFoundError as e:
    print(f"ERROR: Could not find benchmark results. Make sure to run the benchmark cells above first.")
    print(f"   Error: {e}")
except Exception as e:
    print(f"ERROR: Error loading results: {e}")
    import traceback
    traceback.print_exc()

## Part 6: Visualize Performance

Let's create comparison charts.

In [ ]:
# Create comparison visualizations
try:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Chart 1: Throughput comparison
    models = ['Original\n(FP16)', 'Quantized\n(INT4)']
    throughput = [orig_metrics['throughput'], quant_metrics['throughput']]
    
    axes[0].bar(models, throughput, color=['#ee0000', '#00a8e1'])
    axes[0].set_title('Throughput Comparison', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Requests/sec (higher is better)')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Chart 2: Latency comparison
    latency_labels = ['P50', 'P95', 'P99']
    x = range(len(latency_labels))
    width = 0.35
    
    orig_latencies = [orig_metrics['p50_latency'], orig_metrics['p95_latency'], orig_metrics['p99_latency']]
    quant_latencies = [quant_metrics['p50_latency'], quant_metrics['p95_latency'], quant_metrics['p99_latency']]
    
    axes[1].bar([i - width/2 for i in x], orig_latencies, width, label='Original (FP16)', color='#ee0000')
    axes[1].bar([i + width/2 for i in x], quant_latencies, width, label='Quantized (INT4)', color='#00a8e1')
    
    axes[1].set_title('Latency Comparison', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Latency (sec, lower is better)')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(latency_labels)
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('benchmark_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nCharts saved to: benchmark_comparison.png")
    
except Exception as e:
    print(f"ERROR: Could not create charts: {e}")
    print("Make sure you've run the benchmark cells above first.")
    import traceback
    traceback.print_exc()

## Part 7: Model Details

Technical comparison of the two model configurations.

In [ ]:
# Model configuration comparison
import pandas as pd

model_specs = {
    'Specification': [
        'Model Name',
        'Precision',
        'Model Size',
        'Parameters',
        'Quantization Method',
        'Speculative Decoding',
        'Max Context Length'
    ],
    'Original (FP16)': [
        'Qwen/Qwen3.6-35B-A3B',
        'FP16 (16-bit)',
        '~70 GB',
        '35 billion',
        'None',
        'No',
        '8,192 tokens'
    ],
    'Quantized (NVFP4)': [
        'RedHatAI/Qwen3.6-35B-A3B-NVFP4',
        'NVFP4 (4-bit floating point)',
        '~20 GB',
        '35 billion',
        'NVFP4 (4-bit floating point quantization)',
        'MTP (1 token)',
        '8,192 tokens'
    ],
    'Reduction': [
        '—',
        '4x smaller precision',
        '~70% smaller',
        'Same',
        '—',
        '+Speculative',
        'Same'
    ]
}

df = pd.DataFrame(model_specs)
print("\nMODEL CONFIGURATION COMPARISON")
print("=" * 100)
print(df.to_string(index=False))
print("=" * 100)

print("\nKey Insights:")
print("  - Same model architecture (Qwen3.6-35B-A3B), different numeric precision")
print("  - ~70% reduction in model size without changing parameters")
print("  - Quantized model uses MTP speculative decoding for extra speed boost")
print("  - Quantization reduces storage and memory, not parameter count")

## Summary

You've successfully benchmarked both models and compared their performance.

The comparison above shows the throughput and latency differences between the original FP16 model and the quantized INT4 model.

In [ ]:
# Benchmark data is stored in JSON format
print("Benchmark data files:")
print("  - original/benchmarks.json")
print("  - quantized/benchmarks.json")
print("\nYou can download these files to analyze the detailed metrics further.")
print("To download: Right-click the files in the file browser on the left, then select 'Download'.")

## Next Steps

### What You Learned

1. **Quantization fundamentals** - FP16, INT8, INT4 precision formats
2. **Performance benchmarking** - Using guidellm to measure throughput and latency
3. **Real-world trade-offs** - 1.5-2x speedup with minimal quality loss

### Key Takeaways

- Quantized models are **significantly faster** (1.5-2x throughput)
- Latency is **30-40% lower** across P50/P95/P99
- Memory usage is **50-75% less** (allows bigger models on same hardware)
- Quality degradation is typically **<2%** for INT4 quantization

### Try It Yourself

1. **Quantize your own models** with [LLM Compressor](https://github.com/vllm-project/llm-compressor)
2. **Deploy in production** with [vLLM](https://docs.vllm.ai/)
3. **Explore RedHat AI models** at [huggingface.co/RedHatAI](https://huggingface.co/RedHatAI)
4. **Experiment with different quantization levels** (INT8, FP8, structured pruning)

### Resources

- **Workshop Repo:** [github.com/soyr-redhat/boston-tech-week-26-llm-compressor](https://github.com/soyr-redhat/boston-tech-week-26-llm-compressor)
- **vLLM Documentation:** [docs.vllm.ai](https://docs.vllm.ai/)
- **guidellm GitHub:** [github.com/vllm-project/guidellm](https://github.com/vllm-project/guidellm)
- **LLM Compressor:** [github.com/vllm-project/llm-compressor](https://github.com/vllm-project/llm-compressor)

---

**Questions?** Open an issue on GitHub!